<a href="https://colab.research.google.com/github/haze25102583/CNN/blob/main/day2_lesson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[실습1] 정규화, 채널 축 변화
1. 가상 흑백 배치 만들기
2. 픽셀값 0~1로 정규화
3. 맨 뒤에 채널 축 추가
4. 전후 shape & 범위 비교
5. 자동 검사 통과 확인

# CNN 구조와 이미지 분류

#### CNN 핵심 원리

Inductive Bias (귀납 편향)

    1. Local Connectivity
    2. Weight Sharing
    3. 계층적 특징 조합
4. Translation Equivarivance



#### 합성곱 연산 (Conv2D)

1. Filters

        출력 채널 수 => Feature map
        W x H x Channel

2. Kernel

        W x H

3. Padding: Valid vs Same

4. Stride: 이동 간격

5. Feature Map: 필터 반응 지도

6. 파라미터 계산-> 가중치 공유

#### 공간 축소 및 활성화
1. ReLU

        합성곱 결과: 양의 반응-> 그대로, 음의 반응-> 0으로 차단하여
                     다음 층에 넘기지 xx
        다양한 특징을 다양한 층을 거치면서 '비선형적' 패턴을 학습

2. Pooling

        강한 패턴(양의 반응)만 남김-> HxW 축소
        Receptive Field 확장 : 깊은 층의 원본 이미지의 범위(수용 영역)이 확장
        (예) MaxPooling: 영역 최댓값 요약, AveragePooling: 영역 평균값 요약

3. Receptive Field

        층이 깊어질수록, 맵 한 칸의 원본 이미지의 문맥(범위)가 넓어짐.
        ReLU(활성화): 의미있는 강한 양의 반응만 ReLU를 통해 활성화되어, 다음 층으로 전달.
        Pooling: HxW가 축소되고, 활성화된 강한 반응들만 요약


#### 모델 구조 설계
1. 특징 추출기: Conv/ Pool
2. Flatten: 2차원-> 1차원 벡터
3. GAP

        Global Average Pooling.
        파라미터 절감.

4. Dense

    중간 Dense 층

        Dense(64) + ReLU
        앞서 전달받은 특징 벡터들을 서로 가중합-> 새롭게 조정

    최종 Dense + Softmax 출력층

        Softmax 활성화 함수: 출력된 점수들을 합이 1.0되는 확률값으로 변환

#### 데이터 및 학습 분석
1. Input Shape (N, H, W, C)
2. Model Summary()

        마지막 출력층의 뉴런 개수(unit) == 실제 데이터의 정답 클래스 개수

3. 학습 곡선 : 과적합, 과소적합 판단
4. 혼동행렬 : 오분류 분석
5. 정규화 (0~1 범위 변환)

In [ ]:
import numpy as np

x = np.zeros((2, 28, 28), dtype=np.uint8)
x[0,0,0] = 255

scaled = x.astype("float32")/255.0
ready = np.expand_dims(scaled, axis=-1)

print("원본: ", x.shape, x.min(), x.max())                                       # 원본:  (2, 28, 28) 0 255
print("변환: ", ready.min(),ready.max())                                         # 변환:  0.0 1.0

assert ready.shape == (2,28,28,1)
assert ready.min()==0 and ready.max()==1
print("전처리 확인 완료")

원본:  (2, 28, 28) 0 255
변환:  0.0 1.0
전처리 확인 완료


1. 입력 채널 수는 필터의 깊이를 결정
   필터 : 입력의 모든 채널을 함께 계산 ->> feature map 1장
   필터 K개 -> feature map K장 = 출력 채널 K개

2. 필터 깊이 = 입력 채널 수
   흑백 : 3x3x1
   RGB : 3x3x3

3. 정수 라벨
      손실 함수 : sparse_categorical_crossentropy
   원-핫 라벨
      손실 함수 : categorical_crossentropy

4. ANN : CNN의 효과를 판단하는 기준 모델
    2D 위치 관계를 구조로 활용하기 어려움
    국소 패턴마다 별도 가중치 필요

5. Dense, Conv의 파라미터
  Dense(64) : 흑백
    (28x28x1 +1)x64
  Dense(64) : RGB
    (224x224x3 +1)x64
  Conv
    3x3x3 = 가중치/필터
    (가중치 +1)x32 = 896

In [ ]:
dense = (28*28 +1) * 64
conv = (3*3*1 +1)*64
ratio = dense/conv

print("Dense: ", dense)                           # Dense:  50240
print("Conv: ", conv)                             # Conv:  640
print("배수: ", ratio)                            # 배수:  78.5

assert dense == 50240
assert conv == 640
print("파라미터 비교 완료")

Dense:  50240
Conv:  640
배수:  78.5
파라미터 비교 완료


In [ ]:
import numpy as np

patch = np.array([[2,0,1],
                  [1,3,0],
                  [0,2,4]])

kernel = np.array([[1,0,-1],
                   [1,0,-1],
                   [1,0,-1]])

response = np.sum(patch*kernel)
relu = max(0, response)

print("합성곱: ", response)
print("ReLU: ", relu)
assert response == -2 and relu == 0

합성곱:  -2
ReLU:  0


In [ ]:
def conv_params(kernel, channels, filters):
  return (kernel * kernel * channels + 1) * filters

for size in [28, 224]:
  params = conv_params(3,1,8)
  print(f"{size}x{size}:", params)

# 28x28: 80
# 224x224: 80

assert conv_params(3,1,8) == 80
print("입력 크기와 무관하게 80개")

28x28: 80
224x224: 80
입력 크기와 무관하게 80개


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

x = tf.zeros((2, 28, 28, 1))
conv = layers.Conv2D(
    filters=32,
    kernel_size=3,
    padding="valid"                                           # padding="same" : 같은 이미지의 크기를 볼 수 있도록 0을 가지고 있는 열, 행을 추가
)                                                             # padding="valid" : 추가하지 않고 그대로 28 -> 26으로 변한 이유
y = conv(x)

print("입력: ", x.shape)                                      # 입력:  (2, 28, 28, 1)
print("출력: ", y.shape)                                      # 출력:  (2, 26, 26, 32)
assert y.shape[-1] == conv.filters
print("출력 채널: ", y.shape[-1])

입력:  (2, 28, 28, 1)
출력:  (2, 26, 26, 32)
출력 채널:  32


In [ ]:
import math

def output_size(n, kernel, stride, padding=0):
  return math.floor(
      (n +2*padding - kernel) / stride
  ) + 1

out_s1 = output_size(28, 3, 1)
out_s2 = output_size(28, 3, 2)

print("stride 1: ", out_s1)               # stride 1:  26
print("stride 2: ", out_s2)               # stride 2:  13
assert(out_s1, out_s2) == (26, 13)

stride 1:  26
stride 2:  13


In [ ]:
import tensorflow as tf

x = tf.reshape(
    tf.range(1, 17, dtype=tf.float32),
     (1, 4, 4, 1)
)
pool = tf.keras.layers.MaxPooling2D(2)          # MaxPooling2D의 크기 : (2,2)
y = pool(x)

print(tf.squeeze(y).numpy())                    # [[ 6.  8.]
                                                # [14. 16.]]
                                                # tf.squeeze(y)는 y에서 크기가 1인 차원을 제거
print("shape: ", x.shape, "->", y.shape)        # shape:  (1, 4, 4, 1) -> (1, 2, 2, 1)

assert y.shape == (1, 2, 2, 1)
print("Pooling 확인 완료")

[[ 6.  8.]
 [14. 16.]]
shape:  (1, 4, 4, 1) -> (1, 2, 2, 1)
Pooling 확인 완료


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

x = tf.zeros((2, 28, 28, 1))

for padding in ["valid", "same"]:
  conv = layers.Conv2D(
      8, 3, padding=padding
  )
  y = conv(x)
  print(padding, y.shape, conv.count_params())

valid (2, 26, 26, 8) 80
same (2, 28, 28, 8) 80


In [ ]:
import tensorflow as tf

x = tf.zeros((2, 7, 7, 64))
flat = tf.keras.layers.Flatten()(x)
gap = tf.keras.layers.GlobalAveragePooling2D()(x)

flat_dense = (flat.shape[-1] + 1)*64
gap_dense = (gap.shape[-1] + 1)*64

print("Flatten: ", flat.shape, flat_dense)      # Flatten:  (2, 3136) 200768
print("GAP    : ", gap.shape, gap_dense)        # GAP    :  (2, 64) 4160

assert flat.shape[-1] == 3136
assert gap.shape[-1] == 64

Flatten:  (2, 3136) 200768
GAP    :  (2, 64) 4160


In [ ]:
from tensorflow import keras

model = keras.Sequential([
    keras.Input((28, 28, 1)),
    keras.layers.Conv2D(32, 3),
    keras.layers.MaxPooling2D(2),
    keras.layers.Conv2D(64, 3),
    keras.layers.Flatten(),
    keras.layers.Dense(64),
    keras.layers.Dense(10)
])

model.summary()
assert model.count_params() == 515146

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_10 (Conv2D)              │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_6 (Flatten)             │ (None, 7744)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       495,680 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 515,146 (1.97 MB)

 Trainable params: 515,146 (1.97 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
import numpy as np

train = [0.72, 0.82, 0.90, 0.96]
val = [0.70, 0.80, 0.85, 0.83]

best_epoch = int(np.argmax(val)) + 1
final_gap = train[-1] - val[-1]

print("best epoch:", best_epoch)
print("final gap:", round(final_gap, 2))

assert best_epoch == 3
assert np.isclose(final_gap, 0.13)

best epoch: 3
final gap: 0.13


In [ ]:
import numpy as np

prob = np.array(
    [0.02, 0.05, 0.78, 0.10, 0.05]
)

pred_class = int(np.argmax(prob))
confidence = float(np.max(prob))

print("합    :", prob.sum())            # 합    : 1.0
print("클래스:", pred_class)            # 클래스: 2
print("확신도:", confidence)            # 확신도: 0.78

assert np.isclose(prob.sum(), 1.0)
assert pred_class == 2

합    : 1.0
클래스: 2
확신도: 0.78
